# ISTAT parser walkthrough

This notebook is the practical guide for parsing official ISTAT input-output releases in MARIO.


## What this notebook covers

- where the official ISTAT releases come from;
- when to pass one workbook, one extracted release directory, or one zip archive;
- how `IOT` and `SUT` workflows differ;
- how `iot_mode=`, `level=`, `price=`, and `valuation=` are used;
- how `download=True` works with the official ISTAT release pages;
- which caveats matter for this parser.


## Relevant source page

- Official ISTAT tag page: [ISTAT sistema input-output](https://www.istat.it/tag/sistema-input-output/)

This parser targets the official ISTAT release bundles, not arbitrary spreadsheets with a similar shape.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_istat(...)`

The same function supports:

- `IOT` parsing;
- `SUT` parsing;
- local parsing from one workbook, one extracted release directory, or one release zip;
- automatic download of the official release through `download=True`.


## Key arguments

The key public arguments are:

- `path`: local workbook, extracted release directory, release zip, or cache directory;
- `year`: reference year to select from the ISTAT release;
- `table`: choose `"IOT"` or `"SUT"`;
- `iot_mode`: for `IOT`, either `"product"` or `"industry"`;
- `level`: for `SUT`, either `"63"` or `"20"`;
- `price`: for `SUT`, either `"current"` or `"pyp"`;
- `valuation`: for `SUT`, either `"basic"` or `"purchaser"`;
- `extensions`: optional `emissions`, `employment`, `energy` groups for the 63-branch SUT;
- `download`: when `True`, MARIO downloads the official release before parsing it;
- `edition` and `page_url`: downloader selectors for the official ISTAT release page.


## `IOT` versus `SUT`

Use `table="IOT"` when you want the symmetric representation with `Z`, `Y`, and `V`.

Use `table="SUT"` when you want the native split structure with `S`, `U`, `Yc`, `Ya`, `Va`, and `Vc`.

For `IOT`, the key selector is `iot_mode=`. For `SUT`, the key selectors are `level=`, `price=`, and `valuation=`.


## Workbook, extracted directory, or zip archive

For `IOT`, passing the workbook directly is often the simplest workflow.

For `SUT`, passing the extracted release directory or the release zip is usually more convenient, because MARIO resolves the required workbook triplet from the official file names.


## Download workflow

If you already have the official release locally, point the parser to the workbook, directory, or zip file.

If you want MARIO to fetch the official release first, use `download=True` together with a cache directory in `path`.


## Expected local release layout

Typical local structure:

```text
ISTAT/
├── input_output_2015_2020.zip
├── input_output_2020_2022.zip
└── extracted_release/
    ├── *.xlsx
    └── *.xls
```

For local parsing, the workbook names and sheets must come from the official ISTAT release. For `download=True`, `path` is the cache directory where MARIO stores the downloaded archive and extracted files. The direct helper is `mario.download_istat_io(...)`.

In [ ]:
import mario

## Parse one local `IOT` workbook

This is the simplest `IOT` workflow. If you haven't downloaded it, you can indicate a folder path and download=True


In [ ]:
db = mario.parse_istat(
    path="/path/to/IOT",
    year=2022,
    table="IOT",
    iot_mode="product",
    download=True,
    edition="latest",
)


## Parse one `SUT` release

The `SUT` workflow needs the structural selectors explicitly.


In [ ]:
db = mario.parse_istat(
    path="/path/to/SUT",
    year=2022,
    table="SUT",
    level="63",
    price="current",
    valuation="basic",
    download=True,
    edition="latest",
)

db

## First inspection

Once the release is parsed, the normal MARIO inspection helpers work exactly as for the other parsers.


In [ ]:
db.table_type

In [ ]:
db.activities

In [ ]:
db.units

## Caveats

- the parser expects the official ISTAT workbook and release naming conventions;
- `SUT` parsing has more structural selectors than `IOT`, so explicit arguments are usually the better choice;
- `download=True` is optional, but it is often the cleanest workflow when you want a stable local archive of the release.


## Separate SUT imports by commodity
The IMPORT workbook provides imported intermediate and final uses. Monetary flows are in millions of euros. No origin-country detail or environmental inventory is supplied. Published rounding is retained.


In [ ]:
sut = mario.parse_istat(
    "istat_cache", year=2022, table="SUT", level="63",
    valuation="basic", import_representation="separate",
    download=True, edition="2020-2022",
)
sut.configuration


In [ ]:
imported_intermediates = sut.query("IMP_a")
imported_final_demand = sut.query("IMP_Y")
domestic_use = sut.query("U")
final_demand_adjustments = sut.query("VY")
imported_intermediates


## Add optional satellite accounts
These accounts are matched by native codes to NACE A64 (the SUT contains 63 branches, excluding section U). `emissions` loads the 24 AEA indicators, `employment` loads employed persons and the USE workbook hours, and `energy` loads PEFA net domestic energy use. Both employment measures are converted from thousands to persons/hours; energy is in TJ.

Environmental household totals populate `EY`; activity accounts populate `Ea`, while `Ec` is zero. There are no foreign embodied impacts in the imported goods. `AEA | GHG` is already in t CO2-eq and must not be summed with its components. Coefficients use the monetary denominator in millions of euros. Aggregate only after attaching these native accounts.

Eurostat sources: [AEA](https://ec.europa.eu/eurostat/databrowser/view/env_ac_ainah_r2/default/table?lang=en), [employment](https://ec.europa.eu/eurostat/databrowser/view/nama_10_a64_e/default/table?lang=en), [PEFA](https://ec.europa.eu/eurostat/databrowser/view/env_ac_pefa04/default/table?lang=en).


In [ ]:
sut = mario.parse_istat(
    path="istat_cache", year=2022, table="SUT", level="63",
    valuation="basic", import_representation="separate",
    extensions=["emissions", "employment", "energy"],
    download=True, edition="2020-2022",
)
activity_accounts = sut.query("Ea")
household_accounts = sut.query("EY")
greenhouse_gases = activity_accounts.loc["AEA | GHG"]
sut.units["Satellite account"]


### Reuse the archived snapshot offline
The default `extensions=None` loads no satellite data. A single group or `"all"` is accepted. Selected data must cover the same year; missing or confidential observations raise an error. Eurostat cache files are saved under `istat_cache/extensions/`, beside local zip/workbook inputs when those are passed instead of a directory. Preserve them with the monetary release to retain the exact vintage. `overwrite_download=True` refreshes selected sources when downloading.

`edition` selects the monetary release, not the Eurostat vintage. Inspect source update dates, retrieval dates, queries, mappings and flags before comparing releases.


In [ ]:
provenance = sut.info["istat_extensions"]
offline_sut = mario.parse_istat(
    path="istat_cache", year=2022, table="SUT", level="63",
    valuation="basic", import_representation="separate",
    extensions="all", download=False,
)
provenance
